# GigaTIME virtual multiplex immunofluorescence in DLBCL — reproducible pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alshawabkehmd/gigatime-dlbcl/blob/main/notebooks/gigatime_dlbcl_pipeline.ipynb)

This notebook reproduces every number and figure in the study, top to bottom.
It applies **GigaTIME** (a model that turns routine H&E into 21 virtual protein stains)
to the public **DLBCL-Morph** lymphoma cohort and asks two questions:

1. **Is the virtual output biologically plausible?** (face validity, co-expression, cell-of-origin)
2. **Does it predict overall survival?** (Cox screen, cross-validated signature, added value over IPI)

plus one exploratory question: **do double-expressor (MYC+/BCL2+) tumors look different?**

### Three ways to run it
| `MODE` | What it does | Needs | Time |
|---|---|---|---|
| `"synthetic"` | Runs everything on fake data with the same columns. Checks the code works. Figures are stamped *SYNTHETIC*. | nothing | ~2 min |
| `"features"` | Reproduces all statistics and figures from the per-patient feature table. | nothing extra — the tables are included in `data/` (CC BY-NC 4.0) | ~1 min, CPU |
| `"full"` | Runs GigaTIME on every H&E patch first, then everything else. | GPU, Hugging Face access to GigaTIME, DLBCL-Morph patches | ~1–2 h |

**How to use:** run the cells in order. Every code cell has a short note above it saying what it does.
Only Section 1 needs editing. The clinical and feature tables are included in `data/` under CC BY-NC 4.0 — cite DLBCL-Morph (doi:10.7937/NVA3-N783); see `data/README.md`.

Code: https://github.com/Alshawabkehmd/gigatime-dlbcl

## 1. Settings — **edit this cell only**
Choose the mode and where your files are. Everything else is fixed in `gtdlbcl/config.py`
(seed 17, 5-fold CV, elastic-net penalty 0.1 / L1 ratio 0.5, MYC ≥40 % and BCL2 ≥50 %).
`N_PERMUTATIONS` and `N_BOOTSTRAP` control the robustness tests; lower them for a quick look.

In [ ]:
MODE = "synthetic"          # "synthetic" | "features" | "full"

DATA_DIR = "data"           # folder holding the two CSV files (features / full mode)
PATCH_ROOT = "/content/drive/MyDrive/HE"   # full mode: <PATCH_ROOT>/<patient_id>/<patch>.png
MAX_PATCHES_PER_PATIENT = None             # full mode: e.g. 20 for a quick test run, None = all

N_PERMUTATIONS = 1000       # robustness: label shuffles (published: 1000)
N_BOOTSTRAP = 2000          # robustness: bootstrap resamples (published: 2000)

OUT_DIR = "outputs"         # results.json and figures are written here

## 2. Get the code
If this notebook is running outside the repository (for example, opened from Colab),
this clones the repository so the `gtdlbcl` package can be imported. It then installs the
two packages Colab does not ship with. Locally, use `pip install -r requirements.txt` instead.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Alshawabkehmd/gigatime-dlbcl"
IN_COLAB = "google.colab" in sys.modules

if not os.path.isdir("gtdlbcl"):
    if os.path.isdir("../gtdlbcl"):          # running from notebooks/ inside the repo
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "gigatime-dlbcl"], check=True)
        os.chdir("gigatime-dlbcl")

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lifelines==0.30.3"], check=True)

print("working directory:", os.getcwd())

## 3. Imports and environment record
Loads the analysis package, applies the figure style, and records package versions
into `outputs/environment.json` so anyone can see exactly what produced the results.

In [ ]:
import json, platform, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from gtdlbcl import config, data, stats, figures, synthetic
from gtdlbcl import __version__ as GTDLBCL_VERSION

figures.set_style()
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
SYNTHETIC = MODE == "synthetic"

env = {"gtdlbcl": GTDLBCL_VERSION, "python": platform.python_version(), "mode": MODE,
       "seed": config.SEED, "gigatime_commit": config.GIGATIME_COMMIT}
for pkg in ["numpy", "pandas", "scipy", "sklearn", "statsmodels", "lifelines", "matplotlib"]:
    env[pkg] = importlib.import_module(pkg).__version__
json.dump(env, open(os.path.join(OUT_DIR, "environment.json"), "w"), indent=2)
env

## 4. GigaTIME inference *(full mode only — skipped otherwise)*

### 4.1 Connect Google Drive and install GigaTIME
Mounts Drive (where the H&E patches are), then clones GigaTIME **at a fixed commit** so the
model code never changes underneath you.

In [ ]:
if MODE == "full":
    from gtdlbcl import inference
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    inference.install_gigatime()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    print("GigaTIME pinned at", config.GIGATIME_COMMIT[:7])
else:
    print("skipped (MODE =", MODE + ")")

### 4.2 Log in to Hugging Face and load the model
GigaTIME's weights are gated: request access at huggingface.co/prov-gigatime/GigaTIME first.
Store your token as a **Colab secret** named `HF_TOKEN` (key icon in the left sidebar) —
never paste it into the notebook. Checks that a GPU is attached.

In [ ]:
if MODE == "full":
    import torch
    from huggingface_hub import login
    token = None
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    login(token=token or os.environ.get("HF_TOKEN"))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    assert device == "cuda", "No GPU: Runtime > Change runtime type > GPU"
    model = inference.load_model(device)
    print("model loaded on", device)
else:
    print("skipped")

### 4.3 Self-test on one patch
Runs a single patch at native size. DLBCL is a CD20-positive B-cell tumor and has no epithelium,
so virtual **CD20 should be high and cytokeratin (CK) low**. Stop if this does not print PASS.

In [ ]:
if MODE == "full":
    first = next(iter(inference.list_patches(PATCH_ROOT, 1).values()))[0]
    m = inference.infer_patch(model, first, device).reshape(-1, 21).mean(0)
    cd20, ck = m[config.CHANNELS.index("CD20")], m[config.CHANNELS.index("CK")]
    print(f"{first}\nCD20 = {cd20:.3f} | CK = {ck:.3f} | channel average = {m.mean():.3f}")
    print("PASS" if cd20 > m.mean() and ck < m.mean() else "CHECK: unexpected pattern - do not continue")
else:
    print("skipped")

### 4.4 Run every patch
Saves one `.npy` file per patch plus `manifest.csv`. **Resumable**: if Colab disconnects, re-run this
cell and finished patches are skipped. Set `MAX_PATCHES_PER_PATIENT = 20` in Section 1 for a quick trial.

In [ ]:
if MODE == "full":
    from tqdm.auto import tqdm
    NPY_DIR = "gigatime_out"
    manifest, skipped = inference.run_inference(model, PATCH_ROOT, NPY_DIR, device,
                                                MAX_PATCHES_PER_PATIENT, progress=tqdm)
    print(f"{len(manifest)} patches | {manifest.case_id.nunique()} patients | {len(skipped)} skipped")
else:
    print("skipped")

### 4.5 Build the per-patient feature table
Collapses each patient's patches into one row: for every channel the mean, the spread between
patches (std), the 90th percentile and the positive fraction (pixels > 0.5) — 84 features.
Writes `data/gigatime_case_features_rich.csv`, the input to everything below.

In [ ]:
if MODE == "full":
    feat_full = inference.aggregate_features(manifest, progress=tqdm)
    os.makedirs(DATA_DIR, exist_ok=True)
    feat_full.to_csv(os.path.join(DATA_DIR, "gigatime_case_features_rich.csv"), index=False)
    print("saved", feat_full.shape)
else:
    print("skipped")

### 4.6 Virtual stain figure
Shows the patch with the strongest virtual CD20 next to its composite overlay and single channels.
These are **virtual** stains — DLBCL-Morph has no real multiplex staining to compare against.

In [ ]:
if MODE == "full":
    from PIL import Image
    ex = inference.best_example(manifest)
    he = np.asarray(Image.open(ex.png_path).convert("RGB")) / 255.0
    vir = np.load(ex.npy_path).astype(np.float32)
    path = figures.mif_showcase(he, vir, FIG_DIR)
    plt.show()
else:
    print("skipped (needs full-mode output)")

## 5. Load the data

### 5.1 Read the clinical and feature tables
- **synthetic:** generates fake data with the same columns (a double-expressor signal is planted on purpose).
- **features / full:** reads the two CSVs from `DATA_DIR` (included in the repository, so nothing to upload).

In [ ]:
if SYNTHETIC:
    clin_raw, feat = synthetic.make_synthetic(seed=config.SEED)
    os.makedirs(DATA_DIR, exist_ok=True)
    clin_raw.to_csv(os.path.join(DATA_DIR, "synthetic_clinical.csv"), index=False)
    clin = data.load_clinical(os.path.join(DATA_DIR, "synthetic_clinical.csv"))
    feat["case_id"] = feat["case_id"].astype(str)
else:
    need = ["clinical_data_cleaned.csv", "gigatime_case_features_rich.csv"]
    missing = [f for f in need if not os.path.exists(os.path.join(DATA_DIR, f))]
    if missing and IN_COLAB:
        from google.colab import files
        print("Upload:", missing)
        os.makedirs(DATA_DIR, exist_ok=True)
        for name, content in files.upload().items():
            open(os.path.join(DATA_DIR, name.split(" (")[0].replace(" ", "")), "wb").write(content)
    clin = data.load_clinical(os.path.join(DATA_DIR, "clinical_data_cleaned.csv"))
    feat = data.load_features(os.path.join(DATA_DIR, "gigatime_case_features_rich.csv"))

print("clinical:", clin.shape, "| features:", feat.shape)

### 5.2 Patient flow
Counts patients at each step so every *n* in the paper can be traced
(e.g. 209 in the cohort → 170 with images → 160 with double-expressor status **and** cell-of-origin).

In [ ]:
results = {"mode": MODE}
results["cohort_flow"] = data.cohort_flow(feat, clin)
pd.Series(results["cohort_flow"])

## 6. Goal 1 — is the virtual output biologically plausible?

### 6.1 Channel levels (face validity)
Average virtual signal per channel across all patients. In a B-cell lymphoma we expect
**CD20 high** and **cytokeratin (CK) low**, since lymphoma has no epithelium.

In [ ]:
results["face_validity"] = stats.face_validity(feat)
figures.channel_levels(feat, FIG_DIR, SYNTHETIC); plt.show()
{k: v for k, v in results["face_validity"].items() if k != "channel_means"}

### 6.2 Co-expression structure
Correlates every channel with every other across patients and clusters them. Plausible output shows
T-cell markers moving together, myeloid markers moving together, and CK running **opposite** to lymphoid markers.

In [ ]:
corr, results["coexpression"] = stats.coexpression(feat)
figures.coexpression_heatmap(corr, FIG_DIR, SYNTHETIC); plt.show()
results["coexpression"]

### 6.3 Cell-of-origin
Tests whether the virtual features can tell germinal-center from non-germinal-center tumors
(Hans algorithm), using out-of-fold AUC (0.5 = chance).

In [ ]:
results["coo_classification"] = stats.coo_classification(feat, clin)
results["coo_classification"]

## 7. Goal 2 — does the virtual output predict survival?

### 7.1 One feature at a time (univariable Cox, FDR-corrected)
A Cox model for each of the 84 features, then Benjamini–Hochberg correction for testing 84 at once.

In [ ]:
univ_table, results["univariable_cox"] = stats.univariable_cox(feat, clin)
display(univ_table.head(8)) if "display" in dir() else print(univ_table.head(8))
results["univariable_cox"]

### 7.2 All features together (cross-validated GigaTIME signature)
An elastic-net Cox model on all features, scored **out-of-fold** (each patient is scored by a model
that never saw them). Repeated over 5 fold-splits; C-index 0.5 = coin flip.

In [ ]:
results["gigatime_signature"] = stats.gigatime_signature(feat, clin)
results["gigatime_signature"]

### 7.3 Clinical comparators on the same patients
C-index of the IPI score, and an out-of-fold Cox model on age, stage, LDH, ECOG and cell-of-origin.

In [ ]:
results["clinical_and_ipi"] = stats.clinical_and_ipi(feat, clin)
results["clinical_and_ipi"]

### 7.4 Added value over clinical variables (likelihood-ratio test)
Compares *clinical model* vs *clinical model + GigaTIME score*. Repeated over 5 fold-splits because the
result depends on the split. `gigatime_hr` below 1 means the score points the wrong way
(patients it calls high-risk actually die less) — a sign of noise, not added value.

In [ ]:
results["added_value"] = stats.added_value(feat, clin)
pd.DataFrame(results["added_value"]["per_seed"]).T

### 7.5 Survival curves
**Positive control:** IPI risk groups should separate — this proves the cohort *can* show survival differences.
**GigaTIME:** patients split at the median signature score.

In [ ]:
results["km"], km_frame = stats.km_tests(feat, clin)
figures.km_ipi(km_frame, results["km"]["ipi_logrank_p"], FIG_DIR, SYNTHETIC); plt.show()
figures.km_gigatime(km_frame, results["km"], FIG_DIR, SYNTHETIC); plt.show()
results["km"]

### 7.6 Summary figure: C-index comparison
GigaTIME next to the clinical prognosticators, with the chance line. The published morphology value
is shown for reference but is optimism-corrected in-sample, so it is not directly comparable.

In [ ]:
figures.cindex_comparison(results["gigatime_signature"], results["clinical_and_ipi"], FIG_DIR, SYNTHETIC)
plt.show()

## 8. Exploratory — double-expressor tumors

### 8.1 Immune-cold signature
Each channel is converted to its **share of the patient's total virtual signal** (removes the
"overall cellularity" confound), then compared between double-expressor (MYC ≥40 %, BCL2 ≥50 %)
and other tumors with logistic regression **adjusted for cell-of-origin**, FDR-corrected.

In [ ]:
del_table, results["double_expressor"] = stats.double_expressor(feat, clin)
figures.double_expressor_bars(del_table, results["double_expressor"], FIG_DIR, SYNTHETIC); plt.show()
results["double_expressor"]

### 8.2 Is it robust?
Four stress tests: shuffled labels (permutation), resampled patients (bootstrap), dropping one
double-expressor at a time (leave-one-out), and different MYC/BCL2 cut-offs.
These show internal stability only — **external validation in another cohort is still needed**.

In [ ]:
rob = stats.double_expressor_robustness(feat, clin, n_perm=N_PERMUTATIONS, n_boot=N_BOOTSTRAP)
figures.robustness_panel(rob, FIG_DIR, SYNTHETIC); plt.show()
results["double_expressor_robustness"] = {k: v for k, v in rob.items() if not k.startswith("_")}
results["double_expressor_robustness"]

## 9. Save and verify

### 9.1 Save results and compare with the published values
Writes everything to `outputs/results.json`. In `features` mode it also checks the key numbers
against `results/expected_results.json` (the values reported in the study) and prints MATCH or DIFF.

In [ ]:
json.dump(results, open(os.path.join(OUT_DIR, "results.json"), "w"), indent=2, default=str)

def get(d, path):
    for k in path.split("."):
        d = d[k]
    return d

if MODE == "features":
    expected = json.load(open("results/expected_results.json"))
    ok = True
    for key, spec in expected["checks"].items():
        got = get(results, key)
        good = abs(got - spec["value"]) <= spec["tolerance"]
        ok &= good
        print(f"{'MATCH' if good else 'DIFF '}  {key:55s} expected {spec['value']}  got {got}")
    print("\nALL KEY RESULTS REPRODUCED" if ok else "\nSOME RESULTS DIFFER - see above")
else:
    print("saved; verification against published values runs in MODE = 'features'")

### 9.2 Download the outputs (Colab)
Zips `outputs/` (results, environment record, figures) for download.

In [ ]:
import shutil
shutil.make_archive("gigatime_dlbcl_outputs", "zip", OUT_DIR)
if IN_COLAB:
    from google.colab import files
    files.download("gigatime_dlbcl_outputs.zip")
print("gigatime_dlbcl_outputs.zip")

## What the results mean
- **Plausible but not prognostic.** The virtual stains show sensible lineage structure (CD20 high, CK low,
  coherent T-cell/myeloid blocks), yet no feature survives FDR correction for survival, the cross-validated
  signature sits below chance, and its score points the wrong way. Applied to lymphoma without retraining,
  GigaTIME did not carry prognostic information.
- **Double-expressor signal.** Double-expressor tumors show a proportionally immune-cold, high-proliferation
  virtual profile that is independent of cell-of-origin and internally robust — a pilot finding that needs
  external validation.
- **Limitations.** Single tissue-microarray cohort; virtual (not measured) staining with no ground truth;
  out-of-distribution use of a solid-tumor model; 21 double-expressor cases.